# Unofficial LeetArxiv Implementation of the 2004 paper A Low-Memory Parallel Version of Matsuo, Chao and Tsujii’s Algorithm

The original paper was designed around point counting algorithms for algebraic curves.

Our implementation's goal is to test the author's thesis: **their algorithm  (Gaudry-Schost) can detect collisions in arbitrary finite fields.**

Integer and elliptic curve tests are pretty mid. Lol let's try an Elliptic Divisibility Sequence over a finite field.

Complete Walkthrough available [here](https://leetarxiv.substack.com/p/gaudry-schost-collision-algorithm)

## 2.1: Coding the Birthday Paradox

This is section 2.1 in the LeetArxiv walkthrough as [described here](https://leetarxiv.substack.com/p/gaudry-schost-collision-algorithm)

In [4]:
#Birthday Paradox
def BirthdayProblem(days, people):
  rangeStart = days - people + 1
  rangeEnd = days + 1

  #Calculate numerator
  numerator = 1
  for i in range(rangeStart, rangeEnd):
    numerator *= i

  #Find denominator
  denominator = days ** people
  probability = 1 - (numerator / denominator)
  print(probability)

BirthdayProblem(days=365, people = 23)

0.5072972343239854


## 2.2 Gaudry and Schost’s Alternative to the Birthday Paradox

Gaudry-Schost is built on the *ball with replacement paradox* idea as [described here](https://leetarxiv.substack.com/p/gaudry-schost-collision-algorithm)

In [8]:
#Ball with replacement paradox
import random
def BallCollision(setSize):
  oddBalls = []
  evenBalls = []
  collisionFound = False
  totalPicks = 0

  #pick random balls and record until we observe a collision
  while not collisionFound:
    ball = random.randint(0, setSize - 1)
    totalPicks += 1

    # Odd step
    if totalPicks % 2 == 1:
      oddBalls.append(ball)
      if ball in evenBalls:
        collisionFound = True
    # Even step
    if totalPicks % 2 == 0:
      evenBalls.append(ball)
      if ball in oddBalls:
        collisionFound = True


  return totalPicks

averagePicks = 0
trialCount = 1000
for i in range(trialCount):
  averagePicks += BallCollision(setSize = 365)

averagePicks /= trialCount
print(f'On average {averagePicks}')


On average 35.054


# 3.0 Gaudry-Schost Collision Algorithm

We describe the entire algorithm on LeetArxiv

In [13]:
import random

def IsBallDistinguished(ball, modValue=5):
    """Define a ball as distinguished if its hash modulo modValue is 0."""
    return hash(ball) % modValue == 0

def BallCollisionLowMemory(setSize):
    oddDistinguishedBalls = set()
    evenDistinguishedBalls = set()
    totalPicks = 0
    collisionFound = False

    while not collisionFound:
        # Start a new chain from a random ball
        ball = random.randint(0, setSize - 1)
        # walk until we find a distinguished ball
        while True:
            totalPicks += 1
            # Check only distinguished balls
            if IsBallDistinguished(ball):
                if totalPicks % 2 == 1:
                    if ball in evenDistinguishedBalls:
                        collisionFound = True
                    oddDistinguishedBalls.add(ball)
                else:  # even step
                    if ball in oddDistinguishedBalls:
                        collisionFound = True
                    evenDistinguishedBalls.add(ball)
                break

            # Simple pseudo-random walk to pick the next ball
            ball = (hash(ball) + 7) % setSize

    totalDistinguishedBalls = len(oddDistinguishedBalls) + len(evenDistinguishedBalls)
    return totalPicks, totalDistinguishedBalls


trialCount = 1000
totalPicksSum = 0
totalDistinguishedSum = 0

for _ in range(trialCount):
    picks, distinguished = BallCollisionLowMemory(setSize=365)
    totalPicksSum += picks
    totalDistinguishedSum += distinguished

averagePicks = totalPicksSum / trialCount
averageDistinguished = totalDistinguishedSum / trialCount

print(f"Average picks to collision: {averagePicks}")
print(f"Average distinguished balls stored (memory usage): {averageDistinguished}")



Average picks to collision: 47.715
Average distinguished balls stored (memory usage): 14.93
